# SVM & Vectorization Demo

This notebook walks through:
1. **What is Vectorization?** — Turning text into numbers a machine can understand
2. **How the same word gets different vector values** depending on context
3. **How SVM (Support Vector Machine) uses those vectors** to classify text
4. **Visualizing Support Vectors** — See which training samples define the decision boundary

---
## Step 0: Install & Import Libraries
Run this cell first to make sure everything is available.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.svm import SVC
from sklearn.pipeline import make_pipeline
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

print("All imports successful!")

---
## Part 1: Two Sentences, One Common Word

Let's take two simple sentences that share the word **"valve"** but mean very different things:

| Sentence | Intended Category |
|----------|------------------|
| `"valve assembly for compressor unit"` | Compressor Parts |
| `"valve stem seal kit for refrigerant line"` | Refrigerant Parts |

Both contain **"valve"**, but the surrounding words change how the machine sees them.

In [ ]:
# Our two sentences with the common word "valve"
sentence_1 = "valve assembly for compressor unit"
sentence_2 = "valve stem seal kit for refrigerant line"

sentences = [sentence_1, sentence_2]

print("Sentence 1:", sentence_1)
print("Sentence 2:", sentence_2)
print("\nCommon word: 'valve'")

---
## Part 2: CountVectorizer — Simple Word Counting

The simplest approach: count how many times each word appears.

Each sentence becomes a row of numbers — one number per unique word in the entire vocabulary.

In [ ]:
# Step 1: Fit a CountVectorizer on both sentences
count_vec = CountVectorizer()
count_matrix = count_vec.fit_transform(sentences)

# Show the vocabulary (each word gets an index)
vocab = count_vec.get_feature_names_out()
print("Vocabulary (all unique words):")
print(vocab)
print(f"\nTotal unique words: {len(vocab)}")

In [ ]:
# Step 2: Show the vector output for each sentence
count_df = pd.DataFrame(
    count_matrix.toarray(),
    columns=vocab,
    index=["Sentence 1", "Sentence 2"]
)

print("=" * 60)
print("COUNT VECTORIZER OUTPUT")
print("=" * 60)
print(count_df.to_string())
print("\n--- Key Observation ---")
print(f"'valve' column: Sentence 1 = {count_df.loc['Sentence 1', 'valve']}, Sentence 2 = {count_df.loc['Sentence 2', 'valve']}")
print("Both sentences have the SAME value for 'valve' (1).")
print("The difference comes from the OTHER words.")

---
## Part 3: TF-IDF Vectorizer — Weighted Word Importance

**TF-IDF** = Term Frequency × Inverse Document Frequency

- **TF**: How often a word appears in THIS sentence
- **IDF**: How rare the word is across ALL sentences

Words that appear in EVERY sentence (like "valve" and "for") get **lower scores**.  
Words unique to one sentence get **higher scores**.

This is exactly what our SVM production code uses (`TfidfVectorizer` + `SVC`).

In [ ]:
# Step 1: Fit a TF-IDF Vectorizer on both sentences
tfidf_vec = TfidfVectorizer()
tfidf_matrix = tfidf_vec.fit_transform(sentences)

tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray().round(4),
    columns=tfidf_vec.get_feature_names_out(),
    index=["Sentence 1", "Sentence 2"]
)

print("=" * 60)
print("TF-IDF VECTORIZER OUTPUT")
print("=" * 60)
print(tfidf_df.to_string())
print("\n--- Key Observation ---")
print(f"'valve' in Sentence 1: {tfidf_df.loc['Sentence 1', 'valve']:.4f}")
print(f"'valve' in Sentence 2: {tfidf_df.loc['Sentence 2', 'valve']:.4f}")
print(f"'for'   in Sentence 1: {tfidf_df.loc['Sentence 1', 'for']:.4f}")
print(f"'for'   in Sentence 2: {tfidf_df.loc['Sentence 2', 'for']:.4f}")
print("\nShared words ('valve', 'for') get LOWER TF-IDF scores because they appear in both sentences.")
print("Unique words ('compressor', 'refrigerant', etc.) get HIGHER scores — they are more informative.")

---
## Part 4: Side-by-Side Comparison

Let's visually compare how the two vectorizers treat the **same sentences**.

In [ ]:
# Side-by-side comparison for Sentence 1
compare_s1 = pd.DataFrame({
    "Word": vocab,
    "CountVec (Sentence 1)": count_matrix.toarray()[0],
    "TF-IDF (Sentence 1)": tfidf_matrix.toarray()[0].round(4)
})

compare_s2 = pd.DataFrame({
    "Word": vocab,
    "CountVec (Sentence 2)": count_matrix.toarray()[1],
    "TF-IDF (Sentence 2)": tfidf_matrix.toarray()[1].round(4)
})

print("=" * 50)
print("SENTENCE 1: 'valve assembly for compressor unit'")
print("=" * 50)
print(compare_s1.to_string(index=False))

print("\n" + "=" * 50)
print("SENTENCE 2: 'valve stem seal kit for refrigerant line'")
print("=" * 50)
print(compare_s2.to_string(index=False))

print("\n--- Summary ---")
print("CountVec: All present words = 1, absent words = 0 (no weighting)")
print("TF-IDF:   Shared words get LOWER scores, unique words get HIGHER scores")
print("\nThis is WHY TF-IDF helps SVM distinguish between similar descriptions!")

In [ ]:
# Visual bar chart comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# CountVectorizer
x = np.arange(len(vocab))
width = 0.35
axes[0].bar(x - width/2, count_matrix.toarray()[0], width, label='Sentence 1', color='steelblue')
axes[0].bar(x + width/2, count_matrix.toarray()[1], width, label='Sentence 2', color='coral')
axes[0].set_xticks(x)
axes[0].set_xticklabels(vocab, rotation=45, ha='right')
axes[0].set_title('CountVectorizer Output')
axes[0].set_ylabel('Count')
axes[0].legend()

# TF-IDF
axes[1].bar(x - width/2, tfidf_matrix.toarray()[0], width, label='Sentence 1', color='steelblue')
axes[1].bar(x + width/2, tfidf_matrix.toarray()[1], width, label='Sentence 2', color='coral')
axes[1].set_xticks(x)
axes[1].set_xticklabels(vocab, rotation=45, ha='right')
axes[1].set_title('TF-IDF Vectorizer Output')
axes[1].set_ylabel('TF-IDF Score')
axes[1].legend()

plt.suptitle('Same Sentences, Different Vectorization', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Notice: In TF-IDF, shared words ('valve', 'for') have EQUAL but REDUCED scores.")
print("The unique words stand out more — this helps SVM draw a better decision boundary.")

---
## Part 5: Watch the Vector Change When We Add More Documents

TF-IDF scores are **relative to the entire corpus**. Adding more sentences changes the vectors.

Let's see what happens when we add a third sentence.

In [ ]:
# Original: 2 sentences
corpus_small = [
    "valve assembly for compressor unit",
    "valve stem seal kit for refrigerant line"
]

# Expanded: add a third sentence that also mentions "valve"
corpus_expanded = [
    "valve assembly for compressor unit",
    "valve stem seal kit for refrigerant line",
    "valve pressure regulator for cooling system"
]

# Vectorize both
tfidf_small = TfidfVectorizer()
tfidf_expanded = TfidfVectorizer()

matrix_small = tfidf_small.fit_transform(corpus_small)
matrix_expanded = tfidf_expanded.fit_transform(corpus_expanded)

# Show "valve" score for Sentence 1 in both cases
valve_idx_small = list(tfidf_small.get_feature_names_out()).index("valve")
valve_idx_expanded = list(tfidf_expanded.get_feature_names_out()).index("valve")

print("=" * 60)
print("HOW 'valve' SCORE CHANGES FOR SENTENCE 1")
print("=" * 60)
print(f"With 2 sentences: valve = {matrix_small.toarray()[0][valve_idx_small]:.4f}")
print(f"With 3 sentences: valve = {matrix_expanded.toarray()[0][valve_idx_expanded]:.4f}")
print("\nThe score DECREASED because 'valve' now appears in ALL 3 sentences.")
print("TF-IDF automatically reduces the weight of common words.")
print("\nThis is why our production model retrains when new reference data is added —")
print("the vectors shift as the corpus grows.")

---
## Part 6: SVM in Action — Text Classification

Now let's see the full pipeline: **TF-IDF + SVM** classifying product descriptions.

This mirrors what our production code does in Block-2.

In [ ]:
# Sample training data (product descriptions and their categories)
train_descriptions = [
    "compressor assembly valve kit",
    "compressor motor bearing replacement",
    "compressor discharge valve plate",
    "compressor scroll set high pressure",
    "refrigerant valve service port adapter",
    "refrigerant recovery unit hose set",
    "refrigerant charging scale digital",
    "refrigerant leak detector sensor probe",
    "filter drier core replacement cartridge",
    "filter housing gasket seal kit",
    "filter suction line strainer mesh",
    "filter air intake panel washable"
]

train_labels = [
    "Compressor", "Compressor", "Compressor", "Compressor",
    "Refrigerant", "Refrigerant", "Refrigerant", "Refrigerant",
    "Filtration", "Filtration", "Filtration", "Filtration"
]

print("Training Data:")
for desc, label in zip(train_descriptions, train_labels):
    print(f"  [{label:12s}] {desc}")

In [ ]:
# Build the pipeline (same as production: TfidfVectorizer + SVC)
model = make_pipeline(TfidfVectorizer(), SVC(kernel='linear', probability=True))

# Train the model
model.fit(train_descriptions, train_labels)
print("Model trained successfully!")
print(f"Pipeline steps: {[step[0] for step in model.steps]}")

In [ ]:
# Test with new descriptions the model has NEVER seen
test_descriptions = [
    "valve assembly for compressor unit",          # Has 'valve' + 'compressor'
    "valve stem seal kit for refrigerant line",    # Has 'valve' + 'refrigerant'
    "filter replacement cartridge drier",           # filter + drier
    "compressor oil separator element",             # compressor context
    "refrigerant pressure gauge manifold set"       # refrigerant context
]

print("=" * 70)
print("PREDICTIONS ON NEW DESCRIPTIONS")
print("=" * 70)

for desc in test_descriptions:
    prediction = model.predict([desc])[0]
    probabilities = model.predict_proba([desc])[0]
    confidence = max(probabilities) * 100
    
    print(f"\nInput:       '{desc}'")
    print(f"Predicted:   {prediction}")
    print(f"Confidence:  {confidence:.1f}%")
    print(f"All scores:  {dict(zip(model.classes_, [f'{p:.1%}' for p in probabilities]))}")

print("\n" + "=" * 70)
print("Notice: Both sentences contain 'valve', but SVM correctly classifies")
print("them into DIFFERENT categories based on the surrounding context words.")
print("This is the power of TF-IDF + SVM working together.")

---
## Part 7: Visualizing the SVM Decision Boundary & Support Vectors

TF-IDF creates high-dimensional vectors (one dimension per word). To **see** how SVM separates the categories, we reduce to 2D using **PCA** (Principal Component Analysis).

This graph shows:
- **Colored regions** — where SVM assigns each category
- **Training points** — our 12 product descriptions as dots
- **Circled points** — the **Support Vectors** (the critical data points SVM uses to draw the boundary)
- **Star markers** — new test descriptions and where they land

In [ ]:
from sklearn.decomposition import PCA

# --- Extract the trained TF-IDF vectorizer and SVM from the pipeline ---
tfidf_step = model.named_steps['tfidfvectorizer']
svm_step   = model.named_steps['svc']

# Vectorize training data and test data
X_train_tfidf = tfidf_step.transform(train_descriptions)
X_test_tfidf  = tfidf_step.transform(test_descriptions)

# Reduce to 2D with PCA (for visualization only)
pca = PCA(n_components=2)
X_train_2d = pca.fit_transform(X_train_tfidf.toarray())
X_test_2d  = pca.transform(X_test_tfidf.toarray())

# Retrain a fresh SVM on the 2D data so decision boundary is plottable
svm_2d = SVC(kernel='linear', probability=True)
svm_2d.fit(X_train_2d, train_labels)

# --- Build the decision region mesh ---
x_min, x_max = X_train_2d[:, 0].min() - 0.3, X_train_2d[:, 0].max() + 0.3
y_min, y_max = X_train_2d[:, 1].min() - 0.3, X_train_2d[:, 1].max() + 0.3
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300),
                      np.linspace(y_min, y_max, 300))
Z = svm_2d.predict(np.c_[xx.ravel(), yy.ravel()])

# Map labels to integers for contour coloring
label_to_int = {lab: i for i, lab in enumerate(svm_2d.classes_)}
Z_int = np.array([label_to_int[z] for z in Z]).reshape(xx.shape)

# Colors per category
colors = {'Compressor': '#2196F3', 'Refrigerant': '#4CAF50', 'Filtration': '#FF9800'}
cmap_bg = plt.matplotlib.colors.ListedColormap([colors[c] for c in svm_2d.classes_])

# --- Plot ---
fig, ax = plt.subplots(figsize=(12, 8))

# Decision regions (light fill)
ax.contourf(xx, yy, Z_int, alpha=0.15, cmap=cmap_bg, levels=np.arange(len(svm_2d.classes_) + 1) - 0.5)
ax.contour(xx, yy, Z_int, colors='grey', linewidths=0.8, alpha=0.5)

# Training points by category
for label in svm_2d.classes_:
    mask = np.array(train_labels) == label
    ax.scatter(X_train_2d[mask, 0], X_train_2d[mask, 1],
               c=colors[label], edgecolors='black', s=120, linewidths=1,
               label=f'{label} (train)', zorder=5)

# Highlight support vectors with a large ring
sv_indices = svm_2d.support_
ax.scatter(X_train_2d[sv_indices, 0], X_train_2d[sv_indices, 1],
           facecolors='none', edgecolors='red', s=300, linewidths=2.5,
           label='Support Vectors', zorder=6)

# Test points as stars
test_preds = svm_2d.predict(X_test_2d)
for i, (desc, pred) in enumerate(zip(test_descriptions, test_preds)):
    short = desc[:25] + '...' if len(desc) > 25 else desc
    ax.scatter(X_test_2d[i, 0], X_test_2d[i, 1],
               c=colors[pred], marker='*', s=350, edgecolors='black',
               linewidths=1.2, zorder=7)
    ax.annotate(short, (X_test_2d[i, 0], X_test_2d[i, 1]),
                textcoords="offset points", xytext=(8, 8), fontsize=7.5,
                fontstyle='italic', color='#333333')

ax.set_xlabel('PCA Component 1', fontsize=11)
ax.set_ylabel('PCA Component 2', fontsize=11)
ax.set_title('SVM Decision Boundary with Support Vectors\n(TF-IDF vectors reduced to 2D via PCA)',
             fontsize=13, fontweight='bold')
ax.legend(loc='best', fontsize=9, framealpha=0.9)
plt.tight_layout()
plt.show()

# Print support vector details
print("=" * 65)
print("SUPPORT VECTORS — the training samples SVM relies on most")
print("=" * 65)
for idx in sv_indices:
    print(f"  [{train_labels[idx]:12s}]  \"{train_descriptions[idx]}\"")
print(f"\nTotal support vectors: {len(sv_indices)} out of {len(train_labels)} training samples")
print("\nThese are the descriptions closest to the decision boundary.")
print("SVM only needs these points to define the separating hyperplane —")
print("all other training points could be removed without changing the boundary.")

In [ ]:
# --- Per-category support vector breakdown ---
print("=" * 65)
print("SUPPORT VECTORS BY CATEGORY")
print("=" * 65)

sv_labels = [train_labels[i] for i in sv_indices]
for cat in svm_2d.classes_:
    cat_svs = [train_descriptions[i] for i in sv_indices if train_labels[i] == cat]
    print(f"\n  {cat} ({len(cat_svs)} support vector{'s' if len(cat_svs) != 1 else ''}):")
    for desc in cat_svs:
        print(f"    → \"{desc}\"")

# --- Show how a new test point gets classified ---
print("\n" + "=" * 65)
print("HOW A NEW DESCRIPTION GETS CLASSIFIED")
print("=" * 65)

sample_desc = "valve assembly for compressor unit"
sample_vec  = tfidf_step.transform([sample_desc])
sample_2d   = pca.transform(sample_vec.toarray())
sample_pred = svm_2d.predict(sample_2d)[0]
sample_proba = svm_2d.predict_proba(sample_2d)[0]

print(f"\n  Input:  \"{sample_desc}\"")
print(f"\n  Step 1: TF-IDF converts text → {sample_vec.shape[1]}-dimensional vector")
print(f"  Step 2: PCA reduces to 2D → [{sample_2d[0][0]:.4f}, {sample_2d[0][1]:.4f}]")
print(f"  Step 3: SVM checks which side of the boundary this point falls on")
print(f"  Step 4: Predicted category → {sample_pred}")
print(f"\n  Confidence scores:")
for cls, prob in zip(svm_2d.classes_, sample_proba):
    bar = '█' * int(prob * 30)
    print(f"    {cls:12s}  {prob:.1%}  {bar}")

print("\nThe support vectors define WHERE the boundary sits.")
print("New points are classified based on which side they land on.")

---
## Summary

| Concept | What It Does | Example |
|---------|-------------|--------|
| **CountVectorizer** | Counts word occurrences | `"valve" → 1` (same in both sentences) |
| **TF-IDF Vectorizer** | Weights words by rarity | `"valve" → 0.33` (lower because shared) |
| **SVM** | Finds the best dividing line between categories | Separates Compressor vs Refrigerant parts |

### Key Takeaways
1. **Same word, different vectors** — TF-IDF gives context-aware weights; shared words score lower
2. **Vectors change when corpus changes** — Adding more documents shifts all TF-IDF scores
3. **SVM finds the optimal separating hyperplane** — maximizing the margin between categories
4. **Our production pipeline** uses `TfidfVectorizer() + SVC(kernel='linear')` to classify part descriptions into taxonomy nodes